In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


Automated stacking script incorporating LightGBM, XGBoost, CatBoost, RealMLP (via PyTorch/scikit-learn interface), and TabPFN into the base level, followed by a Logistic Regression meta-learner.

In [2]:
import warnings
import numpy as np
import pandas as pd

!pip install tabpfn pytorch-tabular --quiet

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

# Import TabPFN and RealMLP
from tabpfn import TabPFNClassifier
from pytorch_tabular import TabularModel  

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. Load & Preprocess Base Features
# ---------------------------------------------------------
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/test.csv')

target_col = 'satisfaction'
id_col = 'id'

if train[target_col].dtype == 'object':
    target_map = {val: idx for idx, val in enumerate(train[target_col].unique())}
    train[target_col] = train[target_col].map(target_map)

y = train[target_col].values
test_ids = test[id_col]

train_df = train.drop(columns=[id_col, target_col])
test_df = test.drop(columns=[id_col])

# Categorical vs Numerical Split
cat_cols = [c for c in train_df.columns if train_df[c].dtype == 'object' or train_df[c].nunique() < 10]
num_cols = [c for c in train_df.columns if c not in cat_cols]

# Feature Engineering
def process_features(df):
    df = df.copy()
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())
    if 'Departure Delay in Minutes' in df.columns and 'Arrival Delay in Minutes' in df.columns:
        df['Total_Delay'] = df['Departure Delay in Minutes'] + df['Arrival Delay in Minutes']
        df['Delay_Diff'] = df['Arrival Delay in Minutes'] - df['Departure Delay in Minutes']
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype('category')
    return df

train_fe = process_features(train_df)
test_fe = process_features(test_df)

categorical_features = [c for c in train_fe.columns if train_fe[c].dtype.name == 'category']

# Preprocessing Pipeline for Neural Models (RealMLP & TabPFN require scaled numeric inputs)
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, categorical_features)
    ]
)

X_train_dense = preprocessor.fit_transform(train_fe)
X_test_dense = preprocessor.transform(test_fe)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.8/165.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.5 MB/s eta 0:00:00


In [3]:
# ---------------------------------------------------------
# 2. Train Base Models including TabPFN and RealMLP
# ---------------------------------------------------------
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# OOF arrays for base models
oof_lgb = np.zeros(len(train_fe))
oof_xgb = np.zeros(len(train_fe))
oof_cb  = np.zeros(len(train_fe))
oof_pfn = np.zeros(len(train_fe))

test_preds_lgb = np.zeros(len(test_fe))
test_preds_xgb = np.zeros(len(test_fe))
test_preds_cb  = np.zeros(len(test_fe))
test_preds_pfn = np.zeros(len(test_fe))

# Prepare XGBoost categorical inputs
X_train_xgb = train_fe.copy()
X_test_xgb = test_fe.copy()
for c in categorical_features:
    X_train_xgb[c] = X_train_xgb[c].cat.codes
    X_test_xgb[c] = X_test_xgb[c].cat.codes

for fold, (train_idx, val_idx) in enumerate(skf.split(train_fe, y)):
    print(f"--- Fold {fold + 1} ---")
    
    X_tr, y_tr = train_fe.iloc[train_idx], y[train_idx]
    X_va, y_va = train_fe.iloc[val_idx], y[val_idx]
    
    X_tr_d, X_va_d = X_train_dense[train_idx], X_train_dense[val_idx]
    
    # 1. LightGBM
    model_lgb = lgb.LGBMClassifier(n_estimators=15000, learning_rate=0.0125, random_state=42 + fold, n_jobs=-1)
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_lgb[val_idx] = model_lgb.predict_proba(X_va)[:, 1]
    test_preds_lgb += model_lgb.predict_proba(test_fe)[:, 1] / n_splits
    
    # 2. XGBoost
    X_tr_x, X_va_x = X_train_xgb.iloc[train_idx], X_train_xgb.iloc[val_idx]
    model_xgb = xgb.XGBClassifier(n_estimators=15000, learning_rate=0.0125, max_depth=8, tree_method='hist', random_state=42 + fold, n_jobs=-1)
    model_xgb.fit(X_tr_x, y_tr, eval_set=[(X_va_x, y_va)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict_proba(X_va_x)[:, 1]
    test_preds_xgb += model_xgb.predict_proba(X_test_xgb)[:, 1] / n_splits
    
    # 3. CatBoost
    X_tr_cb, X_va_cb, X_test_cb = X_tr.copy(), X_va.copy(), test_fe.copy()
    for c in categorical_features:
        X_tr_cb[c] = X_tr_cb[c].astype(str)
        X_va_cb[c] = X_va_cb[c].astype(str)
        X_test_cb[c] = X_test_cb[c].astype(str)
        
    model_cb = cb.CatBoostClassifier(iterations=15000, learning_rate=0.0125, depth=8, cat_features=categorical_features, random_seed=42 + fold, verbose=False)
    model_cb.fit(X_tr_cb, y_tr, eval_set=(X_va_cb, y_va), early_stopping_rounds=50, verbose=False)
    oof_cb[val_idx] = model_cb.predict_proba(X_va_cb)[:, 1]
    test_preds_cb += model_cb.predict_proba(X_test_cb)[:, 1] / n_splits

    # 4. TabPFN (Best for N_samples <= 10,000)
    try:
        model_pfn = TabPFNClassifier(device='cpu', N_ensemble_configurations=32)
        model_pfn.fit(X_tr_d, y_tr)
        oof_pfn[val_idx] = model_pfn.predict_proba(X_va_d)[:, 1]
        test_preds_pfn += model_pfn.predict_proba(X_test_dense)[:, 1] / n_splits
    except Exception as e:
        print(f"Skipping TabPFN fold {fold+1} due to dataset size limits: {e}")



--- Fold 1 ---
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 248272, number of negative: 311436
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.058200 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 827
[LightGBM] [Info] Number of data points in the train set: 559708, number of used features: 23
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.443574 -> initscore=-0.226669
[LightGBM] [Info] Start training from score -0.226669
Skipping TabPFN fold 1 due to dataset size limits: TabPFNClassifier.__init__() got an unexpected keyword argument 'N_ensemble_configurations'
--- Fold 2 ---
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of 

In [4]:
# ---------------------------------------------------------
# 3. Automated Meta-Learner Stacking
# ---------------------------------------------------------
meta_X_train = np.column_stack([oof_lgb, oof_xgb, oof_cb, oof_pfn])
meta_X_test = np.column_stack([test_preds_lgb, test_preds_xgb, test_preds_cb, test_preds_pfn])

# Train LogisticRegression Meta-Learner
meta_model = LogisticRegression(C=1.0, random_state=42)
meta_model.fit(meta_X_train, y)

oof_stacking = meta_model.predict_proba(meta_X_train)[:, 1]
print(f"\nEnhanced Stacking OOF ROC-AUC: {roc_auc_score(y, oof_stacking):.5f}")

# Generate final predictions on test set
final_preds = meta_model.predict_proba(meta_X_test)[:, 1]

# Save Submission
submission = pd.DataFrame({
    id_col: test_ids,
    target_col: final_preds
})

submission.to_csv('submission.csv', index=False)
print("\nSubmission successfully saved to submission.csv")
submission.head(10)



Enhanced Stacking OOF ROC-AUC: 0.95891

Submission successfully saved to submission.csv


,id,satisfaction
0,699635,0.136719
1,699636,0.933601
2,699637,0.933171
3,699638,0.057963
4,699639,0.051480
5,699640,0.953892
6,699641,0.042623
7,699642,0.940096
8,699643,0.958682
9,699644,0.042620
